In [ ]:
import cv2 as cv
import numpy as np
import matplotlib.pyplot as plt
from glob import glob
import time

# Load images in correct order
filenames = sorted(glob('img/task02/*.jpeg'))
images = [cv.imread(f) for f in filenames]
central_idx = 0

def calculate_angle(H, w, h):
    # Get the corners of the image
    corners = np.float32([[0, 0], [w, 0], [w, h], [0, h]]).reshape(-1, 1, 2)
    
    # Transform corners through homography
    transformed_corners = cv.perspectiveTransform(corners, H)
    
    # Calculate center points
    center_orig = np.float32([w/2, h/2]).reshape(-1, 1, 2)
    center_transform = cv.perspectiveTransform(center_orig, H)
    
    # Calculate vectors
    vec1 = np.array([1, 0])  # Reference vector (horizontal)
    vec2 = center_transform[0][0] - center_orig[0][0]
    
    # Calculate angle
    angle = np.arccos(np.clip(np.dot(vec1, vec2) / 
                     (np.linalg.norm(vec1) * np.linalg.norm(vec2)), -1.0, 1.0))
    return np.degrees(angle)

def detect_keypoints(gray):
    # Shi-Tomasi detector
    maxCorners = 1000  # Aumentado para panoramas
    qualityLevel = 0.01
    minDistance = 10
    corners = cv.goodFeaturesToTrack(gray, maxCorners, qualityLevel, minDistance)
    
    if corners is None:
        return []
        
    keypoints = [cv.KeyPoint(x=float(x), y=float(y), size=7) for [[x, y]] in corners]
    return keypoints

def match_images(img1, img2, descriptor_type='SIFT'):
    # Convert to grayscale
    gray1 = cv.cvtColor(img1, cv.COLOR_BGR2GRAY)
    gray2 = cv.cvtColor(img2, cv.COLOR_BGR2GRAY)
    
    # Detect keypoints using Shi-Tomasi
    kp1 = detect_keypoints(gray1)
    kp2 = detect_keypoints(gray2)
    
    if len(kp1) < 10 or len(kp2) < 10:
        return [], [], []
    
    if descriptor_type == 'SIFT':
        # SIFT descriptor
        sift = cv.SIFT_create(
            nfeatures=0,
            nOctaveLayers=3,
            contrastThreshold=0.04,
            edgeThreshold=10,
            sigma=1.6
        )
        kp1, des1 = sift.compute(gray1, kp1)
        kp2, des2 = sift.compute(gray2, kp2)
        
        # Match features using FLANN
        FLANN_INDEX_KDTREE = 1
        index_params = dict(algorithm=FLANN_INDEX_KDTREE, trees=5)
        search_params = dict(checks=50)
        matcher = cv.FlannBasedMatcher(index_params, search_params)
        
        if des1 is not None and des2 is not None:
            matches = matcher.knnMatch(des1, des2, k=2)
            
            # Apply Lowe's ratio test
            good_matches = []
            for m, n in matches:
                if m.distance < 0.7 * n.distance:
                    good_matches.append(m)
        else:
            good_matches = []
        
    elif descriptor_type == 'ORB':
        # ORB descriptor
        orb = cv.ORB_create()
        kp1, des1 = orb.compute(gray1, kp1)
        kp2, des2 = orb.compute(gray2, kp2)
        
        # Match using Brute Force
        matcher = cv.BFMatcher(cv.NORM_HAMMING, crossCheck=True)
        
        if des1 is not None and des2 is not None:
            matches = matcher.match(des1, des2)
            good_matches = sorted(matches, key=lambda x: x.distance)[:50]
        else:
            good_matches = []
            
    elif descriptor_type == 'BRISK':
        # BRISK descriptor
        brisk = cv.BRISK_create()
        kp1, des1 = brisk.compute(gray1, kp1)
        kp2, des2 = brisk.compute(gray2, kp2)
        
        # Match using Brute Force
        matcher = cv.BFMatcher(cv.NORM_HAMMING, crossCheck=True)
        
        if des1 is not None and des2 is not None:
            matches = matcher.match(des1, des2)
            good_matches = sorted(matches, key=lambda x: x.distance)[:50]
        else:
            good_matches = []
    else:
        return [], [], []
    
    return kp1, kp2, good_matches

def create_panorama(images, central_idx, max_angle=40, descriptor_type='SIFT'):
    start_time = time.time()
    
    # Initialize with central image
    h, w = images[central_idx].shape[:2]
    total_width = w * len(images)
    panorama = np.zeros((h, total_width, 3), dtype=np.uint8)
    
    # Place central image
    x_central = w * central_idx
    panorama[:, x_central:x_central + w] = images[central_idx]
    
    # Stitch left images (reverse order: from central to left)
    H_left = np.eye(3)
    for i in range(central_idx - 1, -1, -1):
        kp1, kp2, matches = match_images(images[i+1], images[i], descriptor_type)
        
        if len(matches) < 10:
            print(f"Not enough matches on images: {i} and {i+1}")
            continue
            
        # Get matching points (swap src and dst for left side)
        dst_pts = np.float32([kp2[m.trainIdx].pt for m in matches]).reshape(-1, 1, 2)
        src_pts = np.float32([kp1[m.queryIdx].pt for m in matches]).reshape(-1, 1, 2)
        
        # Calculate homography (note the swap of src and dst)
        H, _ = cv.findHomography(dst_pts, src_pts, cv.RANSAC, 5.0)
        if H is None:
            print(f"Not Homography on images: {i} and {i+1}")
            continue
        
        # Check angle before applying homography
        temp_H = H_left @ H
        angle = calculate_angle(temp_H, w, h)
        if abs(angle) > max_angle/2:
            print(f"Angle too large ({angle:.1f}°) for image {i}")
            break
            
        H_left = temp_H
        warped = cv.warpPerspective(images[i], H_left, (total_width, h))
        mask = (warped.sum(axis=2) != 0)
        panorama[np.where(mask)] = warped[np.where(mask)]
    
    # Stitch right images
    H_right = np.eye(3)
    for i in range(central_idx + 1, len(images)):
        kp1, kp2, matches = match_images(images[i-1], images[i], descriptor_type)
        
        if len(matches) < 10:
            print(f"Not enough matches for images {i-1} and {i}")
            continue
            
        src_pts = np.float32([kp2[m.trainIdx].pt for m in matches]).reshape(-1, 1, 2)
        dst_pts = np.float32([kp1[m.queryIdx].pt for m in matches]).reshape(-1, 1, 2)
        
        H, _ = cv.findHomography(src_pts, dst_pts, cv.RANSAC, 5.0)
        if H is None:
            print(f"Not Homography on images {i-1} and {i}")
            continue
        
        # Check angle before applying homography
        temp_H = H_right @ H
        angle = calculate_angle(temp_H, w, h)
        if abs(angle) > max_angle/2:
            print(f"Angle too large ({angle:.1f}°) for image {i}")
            break
            
        H_right = temp_H
        warped = cv.warpPerspective(images[i], H_right, (total_width, h))
        mask = (warped.sum(axis=2) != 0)
        panorama[np.where(mask)] = warped[np.where(mask)]
    
    # Crop the empty regions
    gray = cv.cvtColor(panorama, cv.COLOR_BGR2GRAY)
    _, thresh = cv.threshold(gray, 1, 255, cv.THRESH_BINARY)
    coords = cv.findNonZero(thresh)
    x, y, w, h = cv.boundingRect(coords)
    
    end_time = time.time()
    print(f"Panorama {descriptor_type} creation time: {end_time - start_time:.2f} seconds")
    
    return panorama[y:y+h, x:x+w]

# Create and display panoramas with all three descriptors
fig, axes = plt.subplots(3, 1, figsize=(20, 30))

descriptors = ['SIFT', 'ORB', 'BRISK']
for i, descriptor in enumerate(descriptors):
    panorama = create_panorama(images, central_idx, max_angle=40, descriptor_type=descriptor)
    axes[i].imshow(cv.cvtColor(panorama, cv.COLOR_BGR2RGB))
    axes[i].set_title(f'Panorama using Shi-Tomasi + {descriptor}')
    axes[i].axis('off')

plt.tight_layout()
plt.show()